In [ ]:
import os
import sys
import json
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from scipy.stats import entropy
import torch
from torch import nn, optim
from torch.utils.data import DataLoader

In [ ]:
# Seed
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# Global path variables
CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
DATA_PATH = os.path.join(CURRENT_DIR, 'data', 'time_series', 'real', 'ECL')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints', 'time_series', 'real', 'ECL')
OUT_PATH = os.path.join(CURRENT_DIR, 'out', 'time_series', 'real', 'ECL')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)

In [ ]:
# Show info on the terminal about how the execution is going.
VERBOSE = True
# Even if there is a checkpoint, the model is retrained.
FORCE_TRAINING_PRE_XAI = True
FORCE_TRAINING_NN = True
FORCE_TRAINING_POST_XAI = True
# Name of this experiment that will appear in the result files.
SUBFIX_NAME = 'gradient'
IS_TS = True

In [ ]:
# Local libraries
from dataset import SlidingWindowDataset
from models import Trainer, LSTMModel
from dlnr import DLNoiseReduction
from utils import symmetric_mean_absolute_percentage_error

In [ ]:
# Make sure that the GPU is being used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [ ]:
df_data = pd.read_parquet(
    os.path.join(
        DATA_PATH,
        'clean.parquet'
    )
)

df_data.rename(columns={"meantemp": "y"}, inplace=True)

# scale the data
scaler = MinMaxScaler()
df_data = pd.DataFrame(scaler.fit_transform(df_data.values), columns=df_data.columns)

# assert there is no more categorical variables
columnas_categoricas = df_data.select_dtypes(include=['object', 'category']).columns
assert not list(columnas_categoricas)

In [ ]:
## Declare a Neural Network model and prepare the data to train it ##
## ------------------------------------------------------------------------------------------ ##
# divide the data into train/test datasets
input_vars = df_data.columns
X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(
    df_data[input_vars].values, df_data['y'].values, test_size=0.2, shuffle=False
)

# Create the dataloaders
batch_size = 64
window_size = 30
train_dataset = SlidingWindowDataset(
    X_train_nn,
    y_train_nn,
    window_size=window_size,
    future=1
)
val_dataset = SlidingWindowDataset(
    X_test_nn,
    y_test_nn,
    window_size=window_size,
    future=1
)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Create Neural Network model
input_size = X_train_nn.shape[1] # Número de variables de entrada
hidden_size = 128  # Número de neuronas en la capa oculta
output_size = 1  # Predicción de una variable
model = LSTMModel(input_size, hidden_size, output_size).to(device)

# Set model parameters and create the model Trainer object
lr = 0.001
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
denoiser_checkpoint_path = os.path.join(
    CHECKPOINT_PATH,
    'orig',
    'nn_orig.pth'
)

# Define the trainer
trainer_basic = Trainer(
    model=model,
    train_generator=train_dataloader,
    val_generator=val_dataloader,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    epoch_scheduler=None,
    batch_scheduler=None,
    patience=15,
    epochs=500,
    checkpoints_path=denoiser_checkpoint_path
)

## Train the Neural Network or load its weights from a checkpoint ##
## ------------------------------------------------------------------------------------------ ##
if os.path.exists(denoiser_checkpoint_path) and not FORCE_TRAINING_NN:
    model.load_state_dict(torch.load(denoiser_checkpoint_path, weights_only=True))
    if VERBOSE:
        print(
            'Checkpoint loaded for the neural network model from path: ',
            denoiser_checkpoint_path
        )
else:
    model, _, _, _, _ = trainer_basic.fit(verbose=VERBOSE)